In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
##Applying Regression model to predict the churn of customers based on the estimated salary and credit score of the customers.
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [4]:
##Label encoding
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])

In [5]:
##Onehot encoding for geography
ohe = OneHotEncoder(handle_unknown='ignore')
geo_encoded = ohe.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=ohe.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [6]:
##Concatenating the onehot encoded dataframe with the original dataframe and dropping the original 'Geography' column
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [9]:
#Split the data into features and target variable
X = data.drop('Exited', axis=1)
y = data['Exited']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
#Scaling the features
scaler = StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

In [12]:
#save the preprocessed data
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

with open('onehot_encoder.pkl', 'wb') as f:
    pickle.dump(ohe, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [13]:
##Train ANN regression model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [14]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer for regression
]
)

# Compile the model
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])

In [15]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [16]:
##Training the model
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [17]:
##Setting early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [18]:
#Train the model
history = model.fit(
    X_train, y_train, validation_data = (X_test, y_test), epochs=100,callbacks=[early_stopping, tensorboard_callback]
)

Epoch 1/100
250/250 [==============================] - 1s 3ms/step - loss: 0.2402 - mae: 0.2402 - val_loss: 0.2075 - val_mae: 0.2075
Epoch 2/100
250/250 [==============================] - 1s 2ms/step - loss: 0.2017 - mae: 0.2017 - val_loss: 0.1844 - val_mae: 0.1844
Epoch 3/100
250/250 [==============================] - 1s 2ms/step - loss: 0.1833 - mae: 0.1833 - val_loss: 0.1748 - val_mae: 0.1748
Epoch 4/100
250/250 [==============================] - 1s 2ms/step - loss: 0.1744 - mae: 0.1744 - val_loss: 0.1692 - val_mae: 0.1692
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 0.1689 - mae: 0.1689 - val_loss: 0.1669 - val_mae: 0.1669
Epoch 6/100
250/250 [==============================] - 1s 2ms/step - loss: 0.1651 - mae: 0.1651 - val_loss: 0.1632 - val_mae: 0.1632
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 0.1620 - mae: 0.1620 - val_loss: 0.1606 - val_mae: 0.1606
Epoch 8/100
250/250 [==============================] - 0s 2ms/step - 

In [24]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [25]:
%tensorboard  --logdir regressionlogs/fit/20260419-200302

Reusing TensorBoard on port 6006 (pid 9308), started 0:00:10 ago. (Use '!kill 9308' to kill it.)

In [28]:
##Evaluate the model on the test set
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f'Test MAE: {test_mae}')


63/63 [==============================] - 0s 2ms/step - loss: 0.1470 - mae: 0.1470
Test MAE: 0.1469787061214447


In [29]:
model.save('churn_regression_model.h5')